# 01. RandomForest 시간 기반 교차검증

실행 결과는 `results/`에 저장됩니다.

In [ ]:
from pathlib import Path
import sys

experiment_dir = Path.cwd() / "0726" if (Path.cwd() / "0726").exists() else Path.cwd()
if str(experiment_dir) not in sys.path:
    sys.path.insert(0, str(experiment_dir))


In [ ]:
"""운영진 RandomForest를 2022~2024 expanding-window Fold로 평가한다."""

import joblib
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

from common import (
    BASE_CAT_COLS,
    RESULTS_DIR,
    TARGET_COL,
    VALID_YEARS,
    Timer,
    brier_metrics,
    load_train,
    print_metrics,
    save_json,
)


def build_model(features: list[str]) -> Pipeline:
    numeric_cols = [c for c in features if c not in BASE_CAT_COLS]
    preprocessor = ColumnTransformer(
        [
            (
                "cat",
                OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
                BASE_CAT_COLS,
            ),
            ("num", SimpleImputer(strategy="median"), numeric_cols),
        ]
    )
    return Pipeline(
        [
            ("pre", preprocessor),
            (
                "clf",
                RandomForestClassifier(
                    n_estimators=100,
                    max_depth=10,
                    min_samples_leaf=200,
                    n_jobs=-1,
                    random_state=42,
                ),
            ),
        ]
    )


def main() -> None:
    train, features = load_train()
    predictions, targets, years = [], [], []
    fold_results = []

    for valid_year in VALID_YEARS:
        train_mask = train["season"] < valid_year
        valid_mask = train["season"] == valid_year
        model = build_model(features)
        with Timer() as timer:
            model.fit(train.loc[train_mask, features], train.loc[train_mask, TARGET_COL])
            pred = model.predict_proba(train.loc[valid_mask, features])[:, 1]
        y = train.loc[valid_mask, TARGET_COL].to_numpy()
        metrics = brier_metrics(y, pred)
        metrics.update({"valid_year": valid_year, "seconds": timer.seconds})
        fold_results.append(metrics)
        print_metrics(str(valid_year), metrics)
        predictions.append(pred.astype(np.float32))
        targets.append(y.astype(np.int8))
        years.append(np.full(len(y), valid_year, dtype=np.int16))

    all_pred = np.concatenate(predictions)
    all_y = np.concatenate(targets)
    all_year = np.concatenate(years)
    overall = brier_metrics(all_y, all_pred)
    print_metrics("OOF 전체", overall)

    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        RESULTS_DIR / "01_rf_oof.npz", y=all_y, prediction=all_pred, year=all_year
    )
    save_json(
        RESULTS_DIR / "01_rf_metrics.json",
        {"model": "RandomForest baseline", "folds": fold_results, "overall": overall},
    )
    joblib.dump(model, RESULTS_DIR / "01_rf_last_fold.pkl", compress=3)


if __name__ == "__main__":
    main()

